### data injection to vector db 

In [22]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### file loading

In [12]:
### functon to load all pdf from a directory

def process_all_pdf(pdf_dir ):
    all_document = []
    pdf_dir = Path(pdf_dir)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = str(pdf_file)
                doc.type = "pdf"
            
            all_document.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Failed to load {pdf_file.name}: {str(e)}")

    print(f"Total documents: {len(all_document)}")
    return all_document

pdf_documents = process_all_pdf("../data/pdf")
    

Found 1 pdf files

Processing: animal-facts.pdf
Loaded 3 pages from animal-facts.pdf
Total documents: 3


### splitting

In [ ]:
### splitting

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=['\n\n', '\n', ' ', ''],
    )
    split_documents = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_documents)} chunks")

    if split_documents:
        print(f"example")
        print(f"content: {split_documents[0].page_content[:200]}")
        print(f"metadata: {split_documents[0].metadata}")
    
    return split_documents

chunks = split_documents(pdf_documents)

split 3 documents into 6 chunks
example
content: Australia 
 
 
Giant Clam 
The Giant Clam, as the name suggests, is the biggest clam in the world. They average 
about four feet (ask the children if they are taller than the clam) 
These bi-valves, c
metadata: {'producer': 'Acrobat Distiller 8.1.0 (Macintosh)', 'creator': 'Microsoft Word: cgpdftops CUPS filter', 'creationdate': '2009-09-28T10:21:03-07:00', 'author': 'Grace Norman', 'moddate': '2009-09-28T10:21:03-07:00', 'title': 'Microsoft Word - animal-facts.doc', 'source': '..\\data\\pdf\\animal-facts.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


### embedding

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\brainberg\langchain-poc\langchainRagPoc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingsManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully: {self.model_name}. Embeddings dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Failed to load model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded. Please call _load_model() first.")

        print(f"Generating embeddings for: {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingsManager()
embedding_manager



Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 610.25it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully: all-MiniLM-L6-v2. Embeddings dimension: 384


### vector store

In [34]:
class VectorStore:
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = '../data/vector_store'):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._init_store()

    def _init_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name, 
                metadata={'description': 'PDF documents embeddings for RAG'}
            )
            print(f"Vector store initialized successfully for collection: {self.collection_name}")
            print(f"Existing documents count: {self.collection.count()}")
        except Exception as e:
            print(f"Failed to initialize vector store: {str(e)}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Document and embeddings must have the same length")
        
        print(f"Adding {len(documents)} documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents count: {self.collection.count()}")
        except Exception as e:
            print(f"Failed to add documents to vector store: {str(e)}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized successfully for collection: pdf_documents
Existing documents count: 0


In [47]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Generating embeddings for: 6 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.49it/s]


Embeddings generated successfully with shape: (6, 384)
Adding 6 documents to vector store
Successfully added 6 documents to vector store
Total documents count: 12


### rag retriever

In [63]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingsManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0,) -> List[Dict[str, Any]]:
        print(f"retrieving documents for query: '{query}'")
        print(f"top_k: {top_k}, score_threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )
        
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
            
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"no documents found")

            return  retrieved_docs

        except Exception as e:
            print(f"error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever.retrieve('Indian cow')

retrieving documents for query: 'Indian cow'
top_k: 5, score_threshold: 0.0
Generating embeddings for: 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 81.96it/s]

Embeddings generated successfully with shape: (1, 384)
retrieved 2 documents (after filtering)


[{'id': 'doc_db3209b1_3',
  'content': 'The animals live high up in the Himalayan Mountains, and are valued by humans for their \nwarm wool, tasty milk, and their ability to haul heavy loads. \nA male yak averages seven feet tall! \n \n \nIndia \n \nBengal Tiger \nFewer than 200 left in the wild \nThe average Bengal tiger weighs 500 pounds! \nIt is the national animal of India \n \nCow \nThe cow holds a sacred place in Hinduism—in most of India, you cannot eat beef as a \nresult \nCows can roam freely through India—you can see them stopping traffic in massive cities \nlike Delhi! \nCow poop often used as fuel. Light some up and you can cook with it! \n \nElephant \nThe Indian elephant, although smaller than its African cousin, is the largest land animal in \nAsia. \nHumans have trained elephants to work as early as 4,000 years ago! \nGanesh, a well-loved Indian god, takes the form of an elephant.  \n \nIndonesia \n \nBird of Paradise \nThese birds have very fancy and colorful feathers 

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content
answer=rag_simple("facts about indian animals",rag_retriever,llm)
print(answer)

retrieving documents for query: 'facts about indian animals'
top_k: 3, score_threshold: 0.0
Generating embeddings for: 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.71it/s]

Embeddings generated successfully with shape: (1, 384)
retrieved 2 documents (after filtering)


*   **Yak:** Lives high in the Himalayan Mountains, valued for wool, milk, and hauling loads; males average seven feet tall.
*   **Bengal Tiger:** Fewer than 200 left in the wild, weighs 500 pounds, and is India's national animal.
*   **Cow:** Sacred in Hinduism (beef not eaten), roams freely, and its poop is used as fuel.
*   **Indian Elephant:** Largest land animal in Asia, trained by humans for work for 4,000 years, and the god Ganesh takes its form.


In [73]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("Indian animals", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

retrieving documents for query: 'Indian animals'
top_k: 3, score_threshold: 0.1
Generating embeddings for: 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.74it/s]

Embeddings generated successfully with shape: (1, 384)
retrieved 2 documents (after filtering)


Answer: Yak, Bengal Tiger, Cow, Elephant
Sources: [{'source': '..\\data\\pdf\\animal-facts.pdf', 'page': 1, 'score': 0.2969156503677368, 'preview': 'The animals live high up in the Himalayan Mountains, and are valued by humans for their \nwarm wool, tasty milk, and their ability to haul heavy loads. \nA male yak averages seven feet tall! \n \n \nIndia \n \nBengal Tiger \nFewer than 200 left in the wild \nThe average Bengal tiger weighs 500 pounds! \nIt is...'}, {'source': '..\\data\\pdf\\animal-facts.pdf', 'page': 1, 'score': 0.2969156503677368, 'preview': 'The animals live high up in the Himalayan Mountains, and are valued by humans for their \nwarm wool, tasty milk, and their ability to haul heavy loads. \nA male yak averages seven feet tall! \n \n \nIndia \n \nBengal Tiger \nFewer than 200 left in the wild \nThe average Bengal tiger weighs 500 pounds! \nIt is...'}]
Confidence: 0.2969156503677368
Context Preview: The animals live high up in the Himalayan Mountains, and are valued by 

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("indian animals", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

retrieving documents for query: 'indian animals'
top_k: 3, score_threshold: 0.1
Generating embeddings for: 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.83it/s]

Embeddings generated successfully with shape: (1, 384)
retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
The animals live high up in the Himalayan Mountains, and are valued by humans for their 
warm wool, tasty milk, and their ability to haul heavy loads. 
A male yak averages

 seven feet tall! 
 
 
India 
 
Bengal Tiger 
Fewer than 200 left in the wild 
The average Bengal tiger weighs 500 pounds! 
It is the national animal of India 
 
Cow 
The cow holds a sacred place in Hinduism—in most of India, you cannot eat beef as a 
result 
Cows can roam freely through India—you can see them stopping traffic in massive cities 
like Delhi! 
Cow poop often used as fuel. Light some up and you can cook with it! 
 
Elephant 
The Indian elephant, although smaller than its African cousin, is the largest land animal in 
Asia. 
Humans have trained elephants to work as early as 4,000 years ago! 
Ganesh, a well-loved Indian god, takes the form of an elephant.  
 
Indonesia 
 
Bird of Paradise 
These birds have very fancy and colorful feathers 
Many of the males dance for females 
These birds eat mostly fruit

The animals live high up in the Himalayan Mountains, and are valued by humans for their 
warm wool, tasty milk, and their ability to haul heavy loads. 
A male yak averages